## CRISP-DM Phase 4: Modeling — DenseNet-121

Melatih DenseNet-121 menggunakan pipeline `datasets` yang sudah disiapkan di Colab tim. Ikuti cell dari atas ke bawah.

### 1. Siapkan training

Tetapkan seed dan folder untuk menyimpan checkpoint.

In [ ]:
# Menetapkan seed dan folder penyimpanan model.
from datetime import datetime
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
tf.keras.utils.set_random_seed(SEED)

run_dir = Path("/content/drive/MyDrive/FGD6/DenseNet121") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True, exist_ok=True)

### 2. Bangun DenseNet-121

Gunakan DenseNet-121 pretrained ImageNet dengan backbone dibekukan dan classifier 10 kelas.

In [ ]:
# Membuat classifier DenseNet-121 untuk 10 kelas daun.
base_model = tf.keras.applications.DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
features = base_model(inputs, training=False)  # Menjaga BatchNormalization tetap pada mode inference.
features = tf.keras.layers.GlobalAveragePooling2D()(features)
features = tf.keras.layers.Dropout(0.3)(features)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(features)
model = tf.keras.Model(inputs, outputs)

### 3. Latih classifier

Latih classifier dengan backbone yang masih beku.

In [ ]:
# Melatih classifier menggunakan fitur dari DenseNet-121 pretrained.
warmup_checkpoint = run_dir / "warmup_best.weights.h5"
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_warmup = model.fit(
    datasets["DenseNet121"]["train"],
    validation_data=datasets["DenseNet121"]["val"],
    epochs=10,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(warmup_checkpoint),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3),
    ],
)

### 4. Buka blok akhir

Muat checkpoint warm-up, buka blok `conv5_`, bekukan BatchNormalization, lalu compile ulang dengan learning rate lebih kecil.

In [ ]:
# Membuka blok akhir DenseNet-121 untuk fine-tuning.
model.load_weights(warmup_checkpoint)
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = (
        layer.name.startswith("conv5_")
        and not isinstance(layer, tf.keras.layers.BatchNormalization)
    )

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

### 5. Lakukan fine-tuning

Latih blok akhir dan classifier dengan learning rate yang lebih kecil.

In [ ]:
# Melakukan fine-tuning dan menyimpan checkpoint dengan val_loss terbaik.
finetune_checkpoint = run_dir / "finetune_best.weights.h5"
history_finetune = model.fit(
    datasets["DenseNet121"]["train"],
    validation_data=datasets["DenseNet121"]["val"],
    epochs=25,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(finetune_checkpoint),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
    ],
)

### 6. Pilih checkpoint

Pilih bobot dengan `val_loss` terendah dari warm-up dan fine-tuning.

In [ ]:
# Memuat checkpoint dengan val_loss terendah.
warmup_val_loss = min(history_warmup.history["val_loss"])
finetune_val_loss = min(history_finetune.history["val_loss"])

if finetune_val_loss < warmup_val_loss:
    selected_stage = "fine-tuning"
    selected_checkpoint = finetune_checkpoint
    selected_val_loss = finetune_val_loss
else:
    selected_stage = "warm-up"
    selected_checkpoint = warmup_checkpoint
    selected_val_loss = warmup_val_loss

model.load_weights(selected_checkpoint)
print(f"Checkpoint terpilih: {selected_stage} | val_loss: {selected_val_loss:.4f}")